# 503: Temperature Contribution Analysis

This notebook calculates the contribution of different forcing components to temperature change
between peak warming year and 2100.

## Approach

For each scenario with `MAGICC_FLAG="ALL"`, we calculate:
1. **Peak warming year** and **peak warming magnitude** per ensemble member
2. **Warming in 2100** per ensemble member
3. **Component contributions** (delta from peak to 2100)

### Component Lookup Rules

For a scenario `{scenario}` with `MAGICC_FLAG="ALL"`:
- **CO2/GHG**: Look up `{scenario}` with `MAGICC_FLAG=CO2` and `MAGICC_FLAG=GHG`
- **NZCO2** (for ZEC proxy): 
  - If scenario contains `_NZCO2`: use `{scenario}` with `MAGICC_FLAG=CO2`
  - If scenario contains `_NZKyoto`: use `{base}_NZCO2` with `MAGICC_FLAG=CO2` (where base = scenario without `_NZKyoto`)
  - Otherwise: use `{scenario}_NZCO2` with `MAGICC_FLAG=CO2`

### Components Analysed

- **ALL**: Total warming (from MAGICC_FLAG=ALL)
- **CO2_NZCO2**: CO2 warming if emissions held constant at net-zero level (ZEC proxy)
- **CO2_Negative**: Cooling from negative CO2 emissions (CO2 - CO2_NZCO2)
- **NonCO2_GHG**: Non-CO2 greenhouse gas contribution (GHG - CO2)
- **Residual**: Other forcings (ALL - GHG)

## Outputs

- `503_warming_metrics.csv`: Per-run metrics for all scenarios
- `503_waterfall_contributions.png`: Waterfall plots comparing scenario subsets

In [ ]:
import os
from pathlib import Path

import dotenv
import pandas as pd
import numpy as np

import seaborn as sns
from tqdm import tqdm

dotenv.load_dotenv()

OUTPUT_FOLDER = Path(os.environ["OUTPUT_FOLDER"])
INDIVIDUAL_RUNS_DIR = OUTPUT_FOLDER / "individual_runs"

In [ ]:
# Configuration
MANIFEST_FILE = OUTPUT_FOLDER / "individual_runs/manifests/2026-05-27_09-04_manifest.csv"

COMPONENTS = ["ALL", "CO2_NZCO2", "CO2_Negative", "NonCO2_GHG", "Residual"]

## Load Data

In [ ]:
# Load manifest and create lookup dict
manifest = pd.read_csv(MANIFEST_FILE)
manifest["key"] = manifest["model"] + "|" + manifest["scenario"] + "|" + manifest["magicc_flag"]
file_lookup = manifest.set_index("key")["output_filename"].to_dict()

# Get all scenarios with MAGICC_FLAG="ALL" as reference scenarios
all_scenarios = manifest[manifest["magicc_flag"] == "ALL"][["model", "scenario"]].drop_duplicates()
print(f"Total reference scenarios (MAGICC_FLAG=ALL): {len(all_scenarios)}")

In [ ]:
def load_timeseries(filepath: Path) -> pd.DataFrame:
    """Load timeseries with run_id index and integer year columns."""
    df = pd.read_csv(filepath)
    year_cols = [c for c in df.columns if c.isdigit()]
    ts = df.set_index("run_id")[year_cols].astype(float)
    ts.columns = ts.columns.astype(int)
    return ts


def extract_at_year(ts: pd.DataFrame, run_ids: np.ndarray, years: np.ndarray) -> np.ndarray:
    """Vectorized extraction of values at specified years for each run."""
    ts_arr = ts.values
    ts_years = ts.columns.values
    ts_run_ids = ts.index.values
    
    run_to_idx = {r: i for i, r in enumerate(ts_run_ids)}
    year_to_idx = {y: i for i, y in enumerate(ts_years)}
    
    values = np.full(len(run_ids), np.nan)
    for i, (run_id, year) in enumerate(zip(run_ids, years)):
        if run_id in run_to_idx and year in year_to_idx:
            values[i] = ts_arr[run_to_idx[run_id], year_to_idx[year]]
    return values


def get_file_lookup_keys(model: str, scenario: str) -> dict:
    """
    Determine file lookup keys for a scenario.
    
    Returns dict with keys: ALL, CO2, GHG, NZCO2 (value is the lookup key)
    """
    keys = {
        "ALL": f"{model}|{scenario}|ALL",
        "CO2": f"{model}|{scenario}|CO2",
        "GHG": f"{model}|{scenario}|GHG",
    }
    
    # Determine NZCO2 lookup based on scenario name
    if "_NZCO2" in scenario:
        # Scenario is already NZCO2 extended - use its own CO2 flag
        keys["NZCO2"] = f"{model}|{scenario}|CO2"
    elif "_NZKyoto" in scenario:
        # NZKyoto extended - NZCO2 comes from base scenario with _NZCO2 suffix
        base_scenario = scenario.replace("_NZKyoto", "")
        keys["NZCO2"] = f"{model}|{base_scenario}_NZCO2|CO2"
    else:
        # Base scenario - NZCO2 comes from {scenario}_NZCO2
        keys["NZCO2"] = f"{model}|{scenario}_NZCO2|CO2"
    
    return keys


def calculate_peak_warming(ts_all: pd.DataFrame, start_year: int = 2020, end_year: int = 2100) -> pd.DataFrame:
    """Calculate peak warming year and magnitude for each run."""
    year_cols = [y for y in ts_all.columns if start_year <= y <= end_year]
    ts_subset = ts_all[year_cols]
    
    peak_idx = ts_subset.values.argmax(axis=1)
    peak_years = np.array(year_cols)[peak_idx]
    peak_values = ts_subset.values[np.arange(len(ts_subset)), peak_idx]
    
    return pd.DataFrame({
        "run_id": ts_all.index,
        "peak_year": peak_years,
        "peak_value": peak_values
    })

## Process Scenarios

In [ ]:
results = []
missing_components = []

for _, row in tqdm(all_scenarios.iterrows(), total=len(all_scenarios), desc="Processing"):
    model, scenario = row["model"], row["scenario"]
    
    # Get file lookup keys
    keys = get_file_lookup_keys(model, scenario)
    
    # Check which files exist
    files = {k: file_lookup.get(keys[k]) for k in ["ALL", "CO2", "GHG", "NZCO2"]}
    
    # Track missing components
    missing = [k for k in ["ALL", "CO2", "GHG"] if files[k] is None]
    if missing:
        missing_components.append({"model": model, "scenario": scenario, "missing": missing})
        continue  # Skip if required components are missing
    
    has_nzco2 = files["NZCO2"] is not None
    if not has_nzco2:
        missing_components.append({"model": model, "scenario": scenario, "missing": ["NZCO2"]})
    
    # Load timeseries
    ts = {k: load_timeseries(INDIVIDUAL_RUNS_DIR / files[k]) for k in ["ALL", "CO2", "GHG"]}
    if has_nzco2:
        ts["NZCO2"] = load_timeseries(INDIVIDUAL_RUNS_DIR / files["NZCO2"])
    
    # Calculate derived components
    ts["NonCO2_GHG"] = ts["GHG"] - ts["CO2"]
    ts["Residual"] = ts["ALL"] - ts["GHG"]
    if has_nzco2:
        ts["CO2_Negative"] = ts["CO2"] - ts["NZCO2"]
    
    # Calculate peak warming from ALL timeseries
    peaks = calculate_peak_warming(ts["ALL"])
    run_ids = peaks["run_id"].values
    peak_years = peaks["peak_year"].values
    
    # Build result DataFrame
    result = pd.DataFrame({
        "model": model,
        "scenario": scenario,
        "run_id": run_ids,
        "peak_year": peak_years,
        "GSAT|ALL|peak": peaks["peak_value"].values,
        "GSAT|ALL|2100": ts["ALL"][2100].values,
    })
    
    # Extract component values at peak and 2100
    for comp_name, ts_key in [("NonCO2_GHG", "NonCO2_GHG"), ("Residual", "Residual")]:
        result[f"GSAT|{comp_name}|peak"] = extract_at_year(ts[ts_key], run_ids, peak_years)
        result[f"GSAT|{comp_name}|2100"] = ts[ts_key][2100].values
    
    if has_nzco2:
        for comp_name, ts_key in [("CO2_NZCO2", "NZCO2"), ("CO2_Negative", "CO2_Negative")]:
            result[f"GSAT|{comp_name}|peak"] = extract_at_year(ts[ts_key], run_ids, peak_years)
            result[f"GSAT|{comp_name}|2100"] = ts[ts_key][2100].values
    
    results.append(result)

metrics_df = pd.concat(results, ignore_index=True)
print(f"\nProcessed {len(metrics_df):,} runs across {len(results)} scenarios")

In [ ]:
# Report missing components
if missing_components:
    missing_df = pd.DataFrame(missing_components)
    print(f"Scenarios with missing components: {len(missing_df)}")
    print("\nBreakdown by missing component:")
    print(missing_df.explode("missing")["missing"].value_counts())
    print("\nFirst 10 examples:")
    display(missing_df.head(10))
else:
    print("All scenarios have complete component sets.")

## Calculate Deltas

In [ ]:
# Calculate deltas (2100 - peak) for all components
for comp in COMPONENTS:
    peak_col, y2100_col = f"GSAT|{comp}|peak", f"GSAT|{comp}|2100"
    if peak_col in metrics_df.columns:
        metrics_df[f"GSAT|{comp}|delta"] = metrics_df[y2100_col] - metrics_df[peak_col]

# Merge net-zero timings (using base scenario name for extended scenarios)
netzero_timings = pd.read_csv(OUTPUT_FOLDER / "102_netzero_timings.csv")

# Create base_scenario column for merging (strip _NZCO2 and _NZKyoto suffixes)
metrics_df["base_scenario"] = metrics_df["scenario"].str.replace("_NZCO2|_NZKyoto", "", regex=True)

metrics_df = metrics_df.merge(
    netzero_timings[["model", "scenario", "year_netzero_co2", "year_netzero_kyoto_ar6gwp100"]],
    left_on=["model", "base_scenario"],
    right_on=["model", "scenario"],
    how="left",
    suffixes=("", "_nz")
).drop(columns=["scenario_nz"])

print(f"Metrics shape: {metrics_df.shape}")
print(f"Unique scenarios: {metrics_df.groupby(['model', 'scenario']).ngroups}")
metrics_df.head()

In [ ]:
# Save metrics to CSV
metrics_df.to_csv(OUTPUT_FOLDER / "503_warming_metrics.csv", index=False)
print(f"Saved metrics to {OUTPUT_FOLDER / '503_warming_metrics.csv'}")

## Summary Statistics

In [ ]:
# Aggregate by scenario (median only for simplicity)
value_cols = [c for c in metrics_df.columns if c.startswith("GSAT|")]
stats = metrics_df.groupby(["model", "scenario"])[value_cols].median()
print(f"Statistics for {len(stats)} scenarios")
stats.round(3)

In [ ]:
# Summary of delta contributions
delta_cols = [f"GSAT|{c}|delta" for c in COMPONENTS if f"GSAT|{c}|delta" in metrics_df.columns]
delta_summary = stats[delta_cols].describe(percentiles=[0.05, 0.25, 0.5, 0.75, 0.95]).T
delta_summary.index = [c.replace("GSAT|", "").replace("|delta", "") for c in delta_cols]
print("Component contributions (median delta, peak to 2100):")
delta_summary.round(3)

## CO2_NZCO2 Decline vs Net-Zero Year

In [ ]:
import matplotlib.pyplot as plt

# Merge with stats (median CO2_NZCO2 delta and peak warming per scenario)
plot_data = stats[["GSAT|CO2_NZCO2|delta", "GSAT|ALL|peak"]].reset_index().merge(
    netzero_timings[["model", "scenario", "year_netzero_co2"]],
    on=["model", "scenario"],
    how="inner"
).dropna()

# Scatter plot colored by peak warming
fig, ax = plt.subplots(figsize=(8, 5))
scatter = ax.scatter(
    plot_data["year_netzero_co2"],
    plot_data["GSAT|CO2_NZCO2|delta"],
    c=plot_data["GSAT|ALL|peak"],
    cmap="RdYlBu_r",
    alpha=0.7,
    s=25
)
cbar = plt.colorbar(scatter, ax=ax)
cbar.set_label("Peak Warming [°C]")
ax.set_xlabel("Year of Net-Zero CO2")
ax.set_ylabel("CO2_NZCO2 Delta (2100 - Peak) [°C]")
ax.set_title("ZEC Proxy Decline vs Net-Zero CO2 Year")
ax.axhline(0, color="gray", linestyle="--", linewidth=0.8)
ax.grid(alpha=0.3)
plt.tight_layout()

## Percentile of Percentile Statistics

Calculate the median of medians, p67 of p67, p34 of p34, p5 of p5, p95 of p95 across the delta metrics.

In [ ]:
# Define columns and percentiles to calculate
# Include peak and 2100 values, not just deltas
value_cols_for_pctl = [f"GSAT|{c}|delta" for c in COMPONENTS if f"GSAT|{c}|delta" in metrics_df.columns]
value_cols_for_pctl += ["GSAT|ALL|peak", "GSAT|ALL|2100"]

PERCENTILES = {"median": 0.5, "p05": 0.05, "p34": 0.34, "p67": 0.67, "p95": 0.95}

# Calculate per-scenario percentiles for each column
scenario_percentiles = metrics_df.groupby(["model", "scenario"])[value_cols_for_pctl].agg(
    [lambda x, q=q: x.quantile(q) for q in PERCENTILES.values()]
)
scenario_percentiles.columns = [f"{col}|{name}" for col in value_cols_for_pctl for name in PERCENTILES.keys()]

print(f"Calculated percentiles for {len(scenario_percentiles)} scenarios")
scenario_percentiles.head()

In [ ]:
def calc_percentile_of_percentile(scenario_pctl_df, components):
    """Calculate percentile-of-percentile statistics across scenarios.
    
    Includes delta components plus peak and 2100 values.
    """
    pop = {}
    # Delta components
    for comp in components:
        col_base = f"GSAT|{comp}|delta"
        if f"{col_base}|median" in scenario_pctl_df.columns:
            pop[comp] = {
                "median_of_median": scenario_pctl_df[f"{col_base}|median"].median(),
                "p05_of_p05": scenario_pctl_df[f"{col_base}|p05"].quantile(0.05),
                "p34_of_p34": scenario_pctl_df[f"{col_base}|p34"].quantile(0.34),
                "p67_of_p67": scenario_pctl_df[f"{col_base}|p67"].quantile(0.67),
                "p95_of_p95": scenario_pctl_df[f"{col_base}|p95"].quantile(0.95),
            }
    
    # Peak and 2100 values
    for col_name in ["GSAT|ALL|peak", "GSAT|ALL|2100"]:
        if f"{col_name}|median" in scenario_pctl_df.columns:
            key = "peak" if "peak" in col_name else "2100"
            pop[key] = {
                "median_of_median": scenario_pctl_df[f"{col_name}|median"].median(),
                "p05_of_p05": scenario_pctl_df[f"{col_name}|p05"].quantile(0.05),
                "p34_of_p34": scenario_pctl_df[f"{col_name}|p34"].quantile(0.34),
                "p67_of_p67": scenario_pctl_df[f"{col_name}|p67"].quantile(0.67),
                "p95_of_p95": scenario_pctl_df[f"{col_name}|p95"].quantile(0.95),
            }
    
    return pop

# All scenarios
pop_all = calc_percentile_of_percentile(scenario_percentiles, COMPONENTS)
pop_all_df = pd.DataFrame(pop_all).T
pop_all_df.index.name = "Component"

print("Percentile of Percentiles - All Scenarios:")
pop_all_df.round(3)

In [ ]:
# Filter to scenarios that achieve net-zero Kyoto GHG
nz_kyoto_idx = metrics_df[metrics_df["year_netzero_kyoto_ar6gwp100"].notna()][
    ["model", "scenario"]
].drop_duplicates().set_index(["model", "scenario"]).index

scenario_percentiles_nzkyoto = scenario_percentiles.loc[
    scenario_percentiles.index.isin(nz_kyoto_idx)
]

# Net-zero Kyoto scenarios
pop_nzkyoto = calc_percentile_of_percentile(scenario_percentiles_nzkyoto, COMPONENTS)
pop_nzkyoto_df = pd.DataFrame(pop_nzkyoto).T
pop_nzkyoto_df.index.name = "Component"

print(f"Percentile of Percentiles - Net-Zero Kyoto Scenarios (n={len(scenario_percentiles_nzkyoto)}):")
pop_nzkyoto_df.round(3)

## Waterfall Plots

Waterfall plots showing contribution of each component to temperature decline from peak to 2100.

In [ ]:
# Calculate scenario-level medians for peak and 2100 (needed for waterfall endpoints)
value_cols = [c for c in metrics_df.columns if c.startswith("GSAT|")]
scenario_medians = metrics_df.groupby(["model", "scenario"])[value_cols].median()
scenario_medians_nzkyoto = scenario_medians.loc[scenario_medians.index.isin(nz_kyoto_idx)]

print(f"All scenarios: {len(scenario_medians)}")
print(f"  Peak (median of medians): {scenario_medians['GSAT|ALL|peak'].median():.3f} °C")
print(f"  2100 (median of medians): {scenario_medians['GSAT|ALL|2100'].median():.3f} °C")
print(f"\nNet-zero Kyoto scenarios: {len(scenario_medians_nzkyoto)}")
print(f"  Peak (median of medians): {scenario_medians_nzkyoto['GSAT|ALL|peak'].median():.3f} °C")
print(f"  2100 (median of medians): {scenario_medians_nzkyoto['GSAT|ALL|2100'].median():.3f} °C")

In [ ]:
def create_waterfall_plot(pop_stats, title, ax):
    """
    Create a waterfall plot for temperature decline contributions.
    
    Parameters:
    - pop_stats: dict with component -> {median_of_median, p34_of_p34, p67_of_p67, ...}
                 Must include 'peak' and '2100' keys for endpoint bars
    - title: Plot title
    - ax: matplotlib axis
    """
    # Set up fonts
    plt.rcParams['font.family'] = 'Arial'
    plt.rcParams['font.sans-serif'] = ['Arial']
    
    # Colors
    color_start = '#2C5F7C'       # Peak bar - blue
    color_delta = {
        'CO2_NZCO2': '#8B4513',    # ZEC - brown
        'CO2_Negative': '#228B22', # Negative emissions - green  
        'NonCO2_GHG': '#FF8C00',   # Non-CO2 GHG - orange
        'Residual': '#9370DB'      # Residual - purple
    }
    color_end = '#2E8B57'         # 2100 bar - sea green
    
    # Error bar styling
    error_kw = {'ecolor': 'black', 'alpha': 0.5, 'capsize': 3, 'capthick': 1}
    
    # Get peak statistics from pop_stats (now using p34 of p34 and p67 of p67)
    peak_median = pop_stats["peak"]["median_of_median"]
    peak_p34 = pop_stats["peak"]["p34_of_p34"]
    peak_p67 = pop_stats["peak"]["p67_of_p67"]
    
    # Get 2100 statistics from pop_stats
    y2100_median = pop_stats["2100"]["median_of_median"]
    y2100_p34 = pop_stats["2100"]["p34_of_p34"]
    y2100_p67 = pop_stats["2100"]["p67_of_p67"]
    
    # Bar positions
    x_positions = [0, 1, 2, 3, 4, 5]
    labels = ["Peak\n[ALL]", "ZEC*", "CO$_2$\nNeg", "Non-CO$_2$\nGHG", "Resid.", "2100\n[ALL]"]
    
    # Plot peak bar
    ax.bar(0, peak_median, width=0.6, color=color_start,
           yerr=[[peak_median - peak_p34], [peak_p67 - peak_median]], error_kw=error_kw)
    
    # Plot delta bars (stacked waterfall)
    bottom = peak_median
    delta_components = ['CO2_NZCO2', 'CO2_Negative', 'NonCO2_GHG', 'Residual']
    
    for i, comp in enumerate(delta_components):
        if comp in pop_stats:
            delta_med = pop_stats[comp]["median_of_median"]
            delta_p34 = pop_stats[comp]["p34_of_p34"]
            delta_p67 = pop_stats[comp]["p67_of_p67"]
            
            ax.bar(i + 1, delta_med, bottom=bottom, width=0.6, color=color_delta[comp],
                   yerr=[[delta_med - delta_p34], [delta_p67 - delta_med]], error_kw=error_kw)
            bottom += delta_med
    
    # Plot 2100 bar
    ax.bar(5, y2100_median, width=0.6, color=color_end,
           yerr=[[y2100_median - y2100_p34], [y2100_p67 - y2100_median]], error_kw=error_kw)
    
    # Formatting with font preferences
    ax.set_xticks(x_positions)
    ax.set_xticklabels(labels, fontname="Arial Narrow", fontsize=12)
    ax.set_ylabel("GSAT (°C)", fontname="Arial", fontsize=13)
    ax.set_title(title, fontname="Arial", fontsize=15, fontweight="bold", loc="left")
    ax.grid(alpha=0.3, axis='y')
    ax.axhline(y=0, color='gray', linestyle='-', linewidth=0.5)
    
    # Set tick label fonts
    ax.tick_params(axis='both', labelsize=12)
    for label in ax.get_yticklabels():
        label.set_fontname("Arial Narrow")
    
    return ax

In [ ]:
# Create waterfall plots
fig, axes = plt.subplots(1, 2, figsize=(12, 5), sharey=True)

# Plot 1: All scenarios
create_waterfall_plot(
    pop_all,
    f"All Scenarios (n={len(scenario_percentiles)})",
    axes[0]
)

# Plot 2: Net-zero Kyoto scenarios
create_waterfall_plot(
    pop_nzkyoto,
    f"Net-Zero Kyoto GHG Scenarios (n={len(scenario_percentiles_nzkyoto)})",
    axes[1]
)

# Set y-axis limits
axes[0].set_ylim(1, 2)
axes[1].set_ylim(1, 2)

# Common y-axis label only on left
axes[1].set_ylabel("")

plt.tight_layout()
plt.savefig(OUTPUT_FOLDER / "503_waterfall_contributions.png", dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
scenario_med_nzkyoto

In [ ]:
# Create waterfall plots for Base, NZCO2, and NZKyoto scenario subsets

# Categorize scenarios by type
def get_scenario_type(scenario):
    if "_NZCO2" in scenario:
        return "NZCO2"
    elif "_NZKyoto" in scenario:
        return "NZKyoto"
    else:
        return "Base"

# Add scenario type to scenario_percentiles index
scenario_types = pd.Series(
    [get_scenario_type(s) for s in scenario_percentiles.index.get_level_values("scenario")],
    index=scenario_percentiles.index,
    name="scenario_type"
)

# Filter scenario_percentiles by type
scenario_pctl_base = scenario_percentiles[scenario_types == "Base"]
scenario_pctl_nzco2 = scenario_percentiles[scenario_types == "NZCO2"]
scenario_pctl_nzkyoto = scenario_percentiles[scenario_types == "NZKyoto"]

# Calculate pop stats for each group (includes peak and 2100)
pop_base = calc_percentile_of_percentile(scenario_pctl_base, COMPONENTS)
pop_nzco2 = calc_percentile_of_percentile(scenario_pctl_nzco2, COMPONENTS)
pop_nzkyoto_subset = calc_percentile_of_percentile(scenario_pctl_nzkyoto, COMPONENTS)

print(f"Base scenarios: {len(scenario_pctl_base)}")
print(f"NZCO2 scenarios: {len(scenario_pctl_nzco2)}")
print(f"NZKyoto scenarios: {len(scenario_pctl_nzkyoto)}")

In [ ]:
pop_base

In [ ]:
# Create 3-panel waterfall plot for Base, NZCO2, NZKyoto
fig, axes = plt.subplots(1, 3, figsize=(15, 5), sharey=True)

# Plot 1: Base scenarios
create_waterfall_plot(
    pop_base,
    f"a)  Base Scenarios (n={len(scenario_pctl_base)})",
    axes[0]
)

# Plot 2: NZCO2 scenarios
create_waterfall_plot(
    pop_nzco2,
    f"b) CO$_2$ emissions flat-lined after\n     net zero CO$_2$ (n={len(scenario_pctl_nzco2)})",
    axes[1]
)

# Plot 3: NZKyoto scenarios
create_waterfall_plot(
    pop_nzkyoto_subset,
    f"c) Kyoto GHG emissions flat-lined after\n   net zero GHG (n={len(scenario_pctl_nzkyoto)})",
    axes[2]
)

# Set y-axis limits
for ax in axes:
    ax.set_ylim(1, 2.1)

# Common y-axis label only on left
axes[1].set_ylabel("")
axes[2].set_ylabel("")

plt.tight_layout()
plt.savefig(OUTPUT_FOLDER / "503_waterfall_by_scenario_type.png", dpi=300, bbox_inches='tight')
plt.savefig(OUTPUT_FOLDER / "503_waterfall_by_scenario_type.svg", bbox_inches='tight')
print(f"Saved to {OUTPUT_FOLDER / '503_waterfall_by_scenario_type.png'}")
print(f"Saved to {OUTPUT_FOLDER / '503_waterfall_by_scenario_type.svg'}")
plt.show()

In [ ]:
# Create grouped waterfall plot with hatches for scenario types
# Shows waterfall-style stacked reductions (Peak -> deltas -> 2100)
plt.rcParams['font.family'] = 'Arial'
plt.rcParams['font.sans-serif'] = ['Arial']

fig, ax = plt.subplots(figsize=(14, 6))

# Define colors for components
component_colors = {
    'peak': '#2C5F7C',           # Peak bar - blue
    'CO2_NZCO2': '#8B4513',      # ZEC - brown
    'CO2_Negative': '#228B22',   # Negative emissions - green  
    'NonCO2_GHG': '#FF8C00',     # Non-CO2 GHG - orange
    'Residual': '#9370DB',       # Residual - purple
    '2100': '#2E8B57'            # 2100 bar - sea green
}

# Define hatches for scenario types
hatches = {
    'Base': '',        # No hatch
    'NZCO2': '///',    # Forward diagonal
    'NZKyoto': 'xxx'   # Cross hatch
}

# Group data
groups = ['Base', 'NZCO2', 'NZKyoto']
group_pop_stats = {
    'Base': pop_base,
    'NZCO2': pop_nzco2,
    'NZKyoto': pop_nzkyoto_subset
}
group_counts = {
    'Base': len(scenario_pctl_base),
    'NZCO2': len(scenario_pctl_nzco2),
    'NZKyoto': len(scenario_pctl_nzkyoto)
}

# Bar width and positions
bar_width = 0.25
x_positions = [0, 1, 2, 3, 4, 5]
x_base = np.array(x_positions)

# Error bar styling
error_kw = {'ecolor': 'black', 'alpha': 0.5, 'capsize': 3, 'capthick': 1}

delta_components = ['CO2_NZCO2', 'CO2_Negative', 'NonCO2_GHG', 'Residual']

for i, grp in enumerate(groups):
    pop_stats = group_pop_stats[grp]
    x_offset = (i - 1) * bar_width
    
    # Plot Peak bar
    peak_median = pop_stats["peak"]["median_of_median"]
    peak_p34 = pop_stats["peak"]["p34_of_p34"]
    peak_p67 = pop_stats["peak"]["p67_of_p67"]
    
    ax.bar(x_base[0] + x_offset, peak_median, width=bar_width,
           color=component_colors['peak'], hatch=hatches[grp], edgecolor='black', linewidth=0.5,
           yerr=[[peak_median - peak_p34], [peak_p67 - peak_median]], error_kw=error_kw)
    
    # Plot delta bars (stacked waterfall style)
    bottom = peak_median
    for j, comp in enumerate(delta_components):
        if comp in pop_stats:
            delta_med = pop_stats[comp]["median_of_median"]
            delta_p34 = pop_stats[comp]["p34_of_p34"]
            delta_p67 = pop_stats[comp]["p67_of_p67"]
            
            ax.bar(x_base[j + 1] + x_offset, delta_med, bottom=bottom, width=bar_width,
                   color=component_colors[comp], hatch=hatches[grp], edgecolor='black', linewidth=0.5,
                   yerr=[[delta_med - delta_p34], [delta_p67 - delta_med]], error_kw=error_kw)
            bottom += delta_med
    
    # Plot 2100 bar
    y2100_median = pop_stats["2100"]["median_of_median"]
    y2100_p34 = pop_stats["2100"]["p34_of_p34"]
    y2100_p67 = pop_stats["2100"]["p67_of_p67"]
    
    ax.bar(x_base[5] + x_offset, y2100_median, width=bar_width,
           color=component_colors['2100'], hatch=hatches[grp], edgecolor='black', linewidth=0.5,
           yerr=[[y2100_median - y2100_p34], [y2100_p67 - y2100_median]], error_kw=error_kw)

# Create hatch legend
from matplotlib.patches import Patch
hatch_legend = [
    Patch(facecolor='white', edgecolor='black', hatch='', label=f'Base (n={group_counts["Base"]})'),
    Patch(facecolor='white', edgecolor='black', hatch='///', label=f'NZCO2 (n={group_counts["NZCO2"]})'),
    Patch(facecolor='white', edgecolor='black', hatch='xxx', label=f'NZKyoto (n={group_counts["NZKyoto"]})')
]

# Formatting
ax.set_xticks(x_base)
ax.set_xticklabels(["Peak\n[ALL]", "ZEC*", "CO$_2$ Neg", "Non-CO$_2$\nGHG", "Residual", "2100\n[ALL]"], 
                   fontname="Arial Narrow", fontsize=12)
ax.set_ylabel("GSAT (°C)", fontname="Arial", fontsize=13)
ax.set_title("Temperature contributions by scenario type", fontname="Arial", fontsize=15, fontweight="bold", loc="left")
ax.grid(alpha=0.3, axis='y')
ax.axhline(y=0, color='gray', linestyle='-', linewidth=0.5)
ax.set_ylim(1, 2.1)

# Set tick label fonts
ax.tick_params(axis='both', labelsize=12)
for label in ax.get_yticklabels():
    label.set_fontname("Arial Narrow")

# Add legend
ax.legend(handles=hatch_legend, loc='upper right', frameon=False, 
          prop={"family": "Arial Narrow", "size": 12})

plt.tight_layout()
plt.savefig(OUTPUT_FOLDER / "503_waterfall_grouped.png", dpi=300, bbox_inches='tight')
plt.savefig(OUTPUT_FOLDER / "503_waterfall_grouped.svg", bbox_inches='tight')
print(f"Saved to {OUTPUT_FOLDER / '503_waterfall_grouped.png'}")
print(f"Saved to {OUTPUT_FOLDER / '503_waterfall_grouped.svg'}")
plt.show()

In [ ]:
# Summary comparison table
summary = pd.DataFrame({
    "All Scenarios": pop_all_df["median_of_median"],
    "NZ Kyoto": pop_nzkyoto_df["median_of_median"],
    "Difference": pop_nzkyoto_df["median_of_median"] - pop_all_df["median_of_median"]
})

print("Median of Medians Comparison (°C):")
summary.round(3)